# 외삽과 적용 범위 실습

**Extrapolation · Domain of Applicability · 적용 도메인**

학습 데이터가 덮지 않는 영역의 예측. 오차와 불확실성이 크게 늘 수 있어 적용 범위를 함께 밝혀야 한다.

소재 분야에서 이해하기: 학습에 없던 원소가 든 조성의 예측은 참고용으로만 쓴다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 학습 범위 밖에서 무슨 일이 생기나

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

x_train = np.linspace(0.2, 0.6, 40)
y_train = np.sin(2 * np.pi * x_train) + rng.normal(0, 0.03, 40)
x_all = np.linspace(0, 1, 300)
y_all = np.sin(2 * np.pi * x_all)
print('학습 범위 %.2f ~ %.2f, 평가 범위 0.00 ~ 1.00' % (x_train.min(), x_train.max()))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

models = [('linear', LinearRegression()),
          ('random forest', RandomForestRegressor(n_estimators=200, random_state=0)),
          ('MLP', MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=8000, random_state=0)),
          ('GP', GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF(0.15), normalize_y=True, random_state=0))]
plt.plot(x_all, y_all, 'k--', lw=2, label='truth')
for name, model in models:
    model.fit(x_train[:, None], y_train)
    plt.plot(x_all, model.predict(x_all[:, None]), label=name)
plt.axvspan(0.2, 0.6, alpha=0.12, color='green')
plt.legend(); plt.xlabel('x'); plt.ylabel('y'); plt.show()

inside = (x_all >= 0.2) & (x_all <= 0.6)
for name, model in models:
    prediction = model.predict(x_all[:, None])
    print('%-14s 범위 내 MAE %.3f / 범위 밖 MAE %.3f'
          % (name, np.mean(np.abs(prediction - y_all)[inside]), np.mean(np.abs(prediction - y_all)[~inside])))

## 2. 적용 범위를 기계적으로 표시하기

학습 데이터와의 거리로 "여기는 신뢰할 수 없다"를 자동 판정할 수 있습니다.

In [ ]:
from sklearn.neighbors import NearestNeighbors

neighbours = NearestNeighbors(n_neighbors=3).fit(x_train[:, None])
distance, _ = neighbours.kneighbors(x_all[:, None])
threshold = np.percentile(neighbours.kneighbors(x_train[:, None])[0].mean(1), 95)
flagged = distance.mean(1) > threshold
print('적용 범위 밖으로 표시된 비율 %.1f%%' % (100 * flagged.mean()))
plt.plot(x_all, distance.mean(1), label='mean distance to 3 nearest training points')
plt.axhline(threshold, color='r', ls='--', label='threshold from training data')
plt.legend(); plt.xlabel('x'); plt.show()
print('예측을 보고할 때 이 판정을 함께 표시하면 외삽을 결론으로 쓰는 사고를 줄일 수 있습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#extrapolation)을 여세요.